# RAG Demo — Retrieval-Augmented Generation

This notebook demonstrates **Retrieval-Augmented Generation (RAG)** using:
- **Qwen3-8B** served via vLLM on OpenShift AI (KServe)
- Synthetic corporate knowledge base (4 documents)
- Pure Python — no external vector DB needed

## What is RAG?
LLMs have broad knowledge but no access to your private data. RAG solves this:
1. **Indexing** — split documents into chunks, compute vector representations
2. **Retrieval** — find chunks most relevant to the user's question
3. **Generation** — feed retrieved chunks as context to the LLM

We compare answers **without RAG** (model guessing) vs **with RAG** (model grounded in documents).

## 1. Setup

In [1]:
!pip install -q openai scikit-learn httpx

In [2]:
import os, glob, re, warnings
import httpx
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

# Endpoint modelu — fallback na přímý KServe endpoint pokud env var není nastavený
MODEL_URL = os.environ.get("MODEL_URL",
    "https://qwen25-vl-7b-kserve-workload-svc.rhoai-playground.svc.cluster.local:8000/v1")
MODEL_NAME = "qwen25-vl-7b"

http_client = httpx.Client(verify=False)
client = OpenAI(base_url=MODEL_URL, api_key="unused", http_client=http_client)

models = client.models.list()
print(f"Connected to model: {[m.id for m in models.data]}")

Connected to model: ['qwen3-8b']


## 2. Load and chunk documents

We load 4 synthetic documents from `data/` — a fictional company's internal knowledge base:
- **security-policy.txt** — passwords, data classification, incident response
- **product-catalog.txt** — 4 enterprise products with pricing
- **hr-handbook.txt** — PTO, parental leave, learning budget, expenses
- **architecture-decisions.txt** — ADRs for event-driven, CockroachDB, internal LLM

In [3]:
def load_documents(data_dir="data"):
    docs = []
    for path in sorted(glob.glob(os.path.join(data_dir, "*.txt"))):
        with open(path) as f:
            docs.append({"filename": os.path.basename(path), "content": f.read()})
    return docs


def chunk_document(doc, chunk_size=500, overlap=50):
    text = doc["content"]
    paragraphs = [p.strip() for p in re.split(r"\n\n+", text) if p.strip()]
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) > chunk_size and current:
            chunks.append({"text": current.strip(), "source": doc["filename"]})
            words = current.split()
            current = " ".join(words[-overlap // 6 :]) + "\n\n" + para
        else:
            current = (current + "\n\n" + para) if current else para
    if current.strip():
        chunks.append({"text": current.strip(), "source": doc["filename"]})
    return chunks


documents = load_documents()
print(f"Loaded {len(documents)} documents:")
for doc in documents:
    print(f"  {doc['filename']} ({len(doc['content'])} chars)")

all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_document(doc))

print(f"\nTotal chunks: {len(all_chunks)}")
for i, c in enumerate(all_chunks):
    print(f"  [{i:2d}] {c['source']:30s} {c['text'][:70]}...")

Loaded 4 documents:
  architecture-decisions.txt (2107 chars)
  hr-handbook.txt (1880 chars)
  product-catalog.txt (1584 chars)
  security-policy.txt (1767 chars)

Total chunks: 18
  [ 0] architecture-decisions.txt     Nexus Technologies — Architecture Decision Records...
  [ 1] architecture-decisions.txt     Nexus Technologies — Architecture Decision Records

ADR-2024-017: Adop...
  [ 2] architecture-decisions.txt     provides at-least-once delivery guarantees with a 72-hour retention wi...
  [ 3] architecture-decisions.txt     for a separate read-replica fleet and custom replication logic.

ADR-2...
  [ 4] hr-handbook.txt                Nexus Technologies — Employee Handbook (Excerpt)
Revision 2026-Q1

PAI...
  [ 5] hr-handbook.txt                receive a prorated allocation starting from their hire date.

PARENTAL...
  [ 6] hr-handbook.txt                parental leave, all benefits including stock vesting continue unchange...
  [ 7] hr-handbook.txt                (i.e., up to EUR 

## 3. Build retrieval index

We use **TF-IDF** — a lightweight, dependency-free approach well-suited for keyword-rich enterprise docs.  
In production, replace with dense embeddings + a vector database.

In [4]:
chunk_texts = [c["text"] for c in all_chunks]
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = vectorizer.fit_transform(chunk_texts)


def retrieve(query, top_k=3):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = scores.argsort()[-top_k:][::-1]
    return [
        {**all_chunks[i], "score": float(scores[i])}
        for i in top_indices
        if scores[i] > 0.05
    ]


test = retrieve("What is the password policy?")
print("Test: 'What is the password policy?'\n")
for r in test:
    print(f"  [{r['score']:.3f}] {r['source']}: {r['text'][:90]}...\n")

Test: 'What is the password policy?'

  [0.197] security-policy.txt: Nexus Technologies — Information Security Policy
Version 4.2 | Effective: January 2026

1....

  [0.064] hr-handbook.txt: day. Receipts required for all expenses above EUR 25.

WORK FROM HOME
Nexus Technologies o...

  [0.059] hr-handbook.txt: (i.e., up to EUR 4,500 total for AI-related learning).

EXPENSE POLICY
Business travel mus...



## 4. Chat functions — with and without RAG

In [5]:
def ask_without_rag(question):
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        max_tokens=512,
        temperature=0.3,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return resp.choices[0].message.content


def ask_with_rag(question, top_k=3):
    retrieved = retrieve(question, top_k=top_k)
    if not retrieved:
        context_block = "No relevant documents found."
    else:
        context_block = "\n\n---\n\n".join(
            f"[Source: {r['source']}]\n{r['text']}" for r in retrieved
        )

    system = (
        "You are a helpful assistant for Nexus Technologies employees. "
        "Answer based ONLY on the provided context. "
        "If the answer is not in the context, say so. Cite the source document."
    )
    user = f"Context documents:\n\n{context_block}\n\n---\nQuestion: {question}"

    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        max_tokens=512,
        temperature=0.3,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return resp.choices[0].message.content, retrieved


def compare(question):
    print("=" * 80)
    print(f"QUESTION: {question}")
    print("=" * 80)

    print("\n--- WITHOUT RAG (model's training data only) ---")
    print(ask_without_rag(question))

    print("\n--- WITH RAG (grounded in retrieved documents) ---")
    answer, sources = ask_with_rag(question)
    print(answer)

    print("\n--- RETRIEVED SOURCES ---")
    for s in sources:
        print(f"  [{s['score']:.3f}] {s['source']}")
    print()

## 5. Side-by-side comparison

These questions target facts that exist **only** in our synthetic documents.  
The model cannot know them from training — watch the difference.

In [6]:
compare("What is the minimum password length at Nexus Technologies?")

QUESTION: What is the minimum password length at Nexus Technologies?

--- WITHOUT RAG (model's training data only) ---
Nexus Technologies does not publicly disclose specific password length requirements for their systems. However, it is generally recommended to follow standard cybersecurity best practices, which typically suggest a minimum password length of **8 characters**. For enhanced security, many organizations now recommend a minimum of **12 characters** or more.

If you are an employee or user of Nexus Technologies, it is best to consult your internal IT policies or contact your organization's help desk for the specific password requirements applicable to your account.

--- WITH RAG (grounded in retrieved documents) ---
The minimum password length at Nexus Technologies is 16 characters. This requirement is specified in the Information Security Policy document. 

Source document: [security-policy.txt]

--- RETRIEVED SOURCES ---
  [0.281] security-policy.txt
  [0.176] architectur

In [7]:
compare("How much does the Nexus Vault HSM cost and what certifications does it have?")

QUESTION: How much does the Nexus Vault HSM cost and what certifications does it have?

--- WITHOUT RAG (model's training data only) ---
The **Nexus Vault HSM** is a hardware security module (HSM) developed by **Nexus Vault**, a company that specializes in secure key management and cryptographic solutions. However, as of the latest available information (up to 2024), **Nexus Vault** is not a widely recognized or mainstream HSM provider in the same league as companies like **Thales**, **Gemalto**, **Entrust**, or **IBM Security**. This means that **specific pricing and certification details** for the Nexus Vault HSM may not be publicly available or widely documented.

### 1. **Cost of Nexus Vault HSM**
- **Pricing**: The cost of the Nexus Vault HSM is likely to vary depending on the model, features, and deployment requirements. However, **no official pricing information** is publicly available on the Nexus Vault website or through standard market channels.
- **Customization**: If the HS

In [8]:
compare("What is the annual learning budget for AI certifications at Nexus Technologies?")

QUESTION: What is the annual learning budget for AI certifications at Nexus Technologies?

--- WITHOUT RAG (model's training data only) ---
I don't have access to specific internal information about Nexus Technologies, including their annual learning budget for AI certifications. For detailed and accurate information, I recommend reaching out directly to Nexus Technologies' HR department or their learning and development team. They would be best equipped to provide you with the most up-to-date and relevant details.

--- WITH RAG (grounded in retrieved documents) ---
The annual learning budget for AI certifications at Nexus Technologies is EUR 3,000, with an additional 50% top-up for approved AI/ML certifications, bringing the total to up to EUR 4,500. This is specified in the hr-handbook.txt document. 

Source: [hr-handbook.txt](#)

--- RETRIEVED SOURCES ---
  [0.572] hr-handbook.txt
  [0.149] architecture-decisions.txt
  [0.127] product-catalog.txt



In [9]:
compare("Why did Nexus Technologies decide to migrate from PostgreSQL to CockroachDB?")

QUESTION: Why did Nexus Technologies decide to migrate from PostgreSQL to CockroachDB?

--- WITHOUT RAG (model's training data only) ---
Nexus Technologies, like many companies, may choose to migrate from PostgreSQL to CockroachDB for a variety of reasons, often related to scalability, performance, and operational needs. While I don't have specific internal information about Nexus Technologies' decision, here are the common reasons companies migrate from PostgreSQL to CockroachDB:

### 1. **Horizontal Scalability**
- **PostgreSQL** is primarily designed for vertical scaling (adding more power to a single server), while **CockroachDB** is built for **horizontal scaling** (adding more nodes to a cluster).
- If Nexus Technologies needed to scale their database across multiple data centers or cloud regions, CockroachDB's distributed architecture would be a better fit.

### 2. **High Availability and Fault Tolerance**
- **CockroachDB** is designed to be **always-on**, with built-in **replic

## 6. Try your own question

In [10]:
compare("How long is parental leave for primary caregivers?")

QUESTION: How long is parental leave for primary caregivers?

--- WITHOUT RAG (model's training data only) ---
Parental leave duration for primary caregivers varies depending on the country, employer policies, and the specific type of leave (e.g., maternity, paternity, or parental leave). Here's a general overview:

### **1. Maternity Leave (for mothers):**
- **United States:** No federal mandate, but some states and employers offer up to 12 weeks of unpaid leave under the **Family and Medical Leave Act (FMLA)**. Some employers may offer paid leave or additional time.
- **Canada:** 15 weeks of unpaid leave (under the **Employment Insurance Act**), with some provinces offering additional paid leave.
- **United Kingdom:** 52 weeks of unpaid leave (under the **Parental Leave and Time Off Act**), with some employers offering paid leave.
- **Germany:** 14 weeks of maternity leave (paid), with additional leave for fathers and parents.
- **France:** 16 weeks of maternity leave (paid), with ad

## Key Takeaways

| | Without RAG | With RAG |
|---|---|---|
| **Accuracy** | Hallucinates plausible but wrong facts | Grounded in actual documents |
| **Sources** | None | Cites specific documents |
| **Private data** | No access | Retrieved at query time |
| **Freshness** | Frozen at training cutoff | Uses latest indexed docs |

### Production next steps
- Dense embeddings via `granite-embedding-125m-english` (LlamaStack)
- Vector database (Milvus, pgvector) for scale
- Reranking for precision
- Guardrails against prompt injection
- All running on **OpenShift AI** — same cluster, same governance